# 01. Mutable defaults

**Tier:** Intermediate. **Why you'd use this:** A helper should append to a list the caller passes in, or start a fresh list when they pass nothing. A default of `[]` looks right and silently shares one list across calls.

**After this lesson you can:** spot a shared mutable default; write `None` and create the list inside the body.
About 25 minutes. **Uses:** [Arguments](../05-functions/02-arguments-lesson.ipynb)


## Quick review

1. When is a function default expression evaluated?
2. Why can two names see the same list mutate?

<details><summary>Answers</summary>

Once, when the `def` runs. Assignment shares the object; mutation changes that one object.
</details>


## The idea

Default values are bound once, when the function is defined, and stored on the function object. If that value is a mutable list or dict, every call that uses the default shares it. The fix is a default of `None`, then create a new list inside the body when the argument is missing.

<details><summary>Explain more</summary>

`append_one.__defaults__` shows the stored default tuple. After two calls with no argument, that same list holds both appends.
</details>


## See it work

Append with a safe default.


In [1]:
def append_one(lst=None):
    # 1. Create a fresh list only when the caller omitted one.
    if lst is None:
        lst = []
    lst.append(1)
    return lst


print(append_one())
print(append_one())
print(append_one([10]))


[1]
[1]
[10, 1]


## Check yourself

Answer these before you run the cell.

1. What does the second `broken()` return?
2. What does `broken.__defaults__` show after those calls?


In [2]:
def broken(lst=[]):
    lst.append(1)
    return lst


print(broken())
print(broken())
print(broken.__defaults__)


[1]
[1, 1]
([1, 1],)


<details><summary>Answers</summary>

`[1, 1]`. The defaults tuple still holds that same growing list.
</details>


## Why it works this way

A `def` statement runs like any other statement. Default expressions are evaluated then, and the results are stored on the function as `__defaults__`. Each later call that omits the argument reuses those stored objects. When the stored object is a list, `append` mutates it in place, so the next call sees the previous appends. Deep dive: [Mutability and aliasing](../../internals/02-mutability/01-aliasing-lesson.ipynb).

The design of defaults is "compute once for speed and fixed defaults," which is perfect for immutable values like numbers and strings. It is the wrong shape for a fresh empty container on every call. The `None` sentinel is the idiom: detect omission without sharing a container. Inside the body, `lst = []` creates a new list for that call only.

Cost is one attribute load of `__defaults__` per omitted argument. The bug's cost is correctness: state leaks across calls that look independent. Evidence is the shared list identity across calls and the growing `__defaults__[0]`. Never "reset" by clearing the default list; callers who passed their own list would be cleared too if you cleared the wrong object.

Immutable defaults are fine. Mutable defaults are a pitfall unless you intend to share a registry on purpose and document that. When you read someone else's function, check `__defaults__` before you trust a bare call to be independent. Tests that call the function twice without arguments are the cheapest detector for this bug. The mechanism is aliasing plus a one-time default evaluation. Once you see that pair, the fix of `None` plus an inner constructor is obvious and stable.


In [3]:
def broken(lst=[]):
    lst.append(1)
    return lst


first = broken()
second = broken()
print(first)
print(second)
print(first is second)
print(broken.__defaults__)


[1, 1]
[1, 1]
True
([1, 1],)


## When to use it

Use `None` (or another immutable sentinel) when the real default is a new list or dict. Use a mutable default only when you intentionally share one registry and say so. Don't clear a default list to "reset" between tests without understanding who holds it. If a test fails only on the second call with no arguments, suspect a shared default before you suspect the caller.


## Common mistake

Writing `def f(items=[])` and expecting a fresh list each call. The second call keeps the first call's appends.


In [4]:
def broken(lst=[]):
    lst.append(1)
    return lst


print(broken())
print(broken())


[1]
[1, 1]


Default to `None` and build the list in the body.


In [5]:
def append_one(lst=None):
    if lst is None:
        lst = []
    lst.append(1)
    return lst


print(append_one())
print(append_one())


[1]
[1]


## Try it

Change `append_one` so a caller can pass an existing list and still see the append, while two bare calls stay independent. Which object identity check proves the bare calls do not share?


## Recap

- Defaults are evaluated once, at `def` time.
- A mutable default is shared across calls.
- Prefer `None` and create the container in the body.

**Next:** [practice](01-mutable-defaults-practice.ipynb)
